In [1]:
# Подключаем Google Диск к Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Загружаем данные
attendance_all = pd.read_csv('/content/drive/MyDrive/проект аналитика/attendance_all.csv')
attendance_filtered = pd.read_csv('/content/drive/MyDrive/проект аналитика/attendance_for_studentTable.csv')
grades = pd.read_csv('/content/drive/MyDrive/проект аналитика/export_studs_cleaned.csv')

print("Файлы загружены!")

# Просмотр структуры данных
print("=== ИНФОРМАЦИЯ О ФАЙЛАХ ===\n")

print("attendance_all - структура:")
print(f"Количество записей: {len(attendance_all)}")
print(f"Колонки: {attendance_all.columns.tolist()}")
print(f"Пропуски в value: {attendance_all['value'].isna().sum()}")
print(f"Уникальных студентов (mira_id): {attendance_all['mira_id'].nunique()}")
print(f"Уникальных дисциплин: {attendance_all['discipline_id'].nunique()}")
print("\n")

print("attendance_filtered - структура:")
print(f"Количество записей: {len(attendance_filtered)}")
print(f"Колонки: {attendance_filtered.columns.tolist()}")
print(f"Пропуски в value: {attendance_filtered['value'].isna().sum()}")
print(f"Уникальных студентов: {attendance_filtered['mira_id'].nunique()}")
print(f"Уникальных дисциплин: {attendance_filtered['discipline_id'].nunique()}")
print("\n")

print("grades - структура:")
print(f"Количество записей: {len(grades)}")
print(f"Колонки: {grades.columns.tolist()}")
print(f"Уникальных студентов (Student_ID): {grades['Student_ID'].nunique()}")
print(f"Уникальных дисциплин: {grades['Discipline_ID'].nunique()}")

Файлы загружены!
=== ИНФОРМАЦИЯ О ФАЙЛАХ ===

attendance_all - структура:
Количество записей: 2235934
Колонки: ['id', 'lesson_id', 'value', 'mira_id', 'discipline_id', 'discipline']
Пропуски в value: 446628
Уникальных студентов (mira_id): 17752
Уникальных дисциплин: 2325


attendance_filtered - структура:
Количество записей: 1154407
Колонки: ['id', 'lesson_id', 'value', 'mira_id', 'discipline_id', 'discipline']
Пропуски в value: 63357
Уникальных студентов: 9446
Уникальных дисциплин: 1787


grades - структура:
Количество записей: 465637
Колонки: ['Faculty', 'Faculty_ID', 'Speciality', 'Speciality_ID', 'Group', 'Student_ID', 'Birthday', 'Is_Academic', 'Discipline_ID', 'Discipline', 'Result_ID', 'Result']
Уникальных студентов (Student_ID): 13220
Уникальных дисциплин: 4666


In [4]:
# 1. СТАТИСТИКА ПО ПОСЕЩАЕМОСТИ ДЛЯ ВСЕХ СТУДЕНТОВ (attendance_all)

print("=== СТАТИСТИКА ПОСЕЩАЕМОСТИ (ВСЕ СТУДЕНТЫ) ===\n")

# Общая статистика по предметам
attendance_stats_all = attendance_all.groupby(['discipline_id', 'discipline']).agg({
    'mira_id': ['nunique', 'count'],  # уникальных студентов и всего записей
    'value': lambda x: (x == 1).sum()  # количество посещений (где value = 1)
}).round(2)

# Переименуем колонки для понятности
attendance_stats_all.columns = ['уникальных_студентов', 'всего_записей', 'количество_посещений']
attendance_stats_all = attendance_stats_all.reset_index()

# Добавим процент посещаемости
attendance_stats_all['процент_посещений'] = (
    (attendance_stats_all['количество_посещений'] / attendance_stats_all['всего_записей'] * 100)
    .round(1)
)

# Сортируем по количеству студентов
attendance_stats_all = attendance_stats_all.sort_values('уникальных_студентов', ascending=False)

print("ТОП-10 предметов по количеству студентов:")
print(attendance_stats_all.head(10).to_string(index=False))
print("\n")

print("Предметы с наибольшим процентом посещаемости:")
print(attendance_stats_all.sort_values('процент_посещений', ascending=False).head(10).to_string(index=False))
print("\n")

print("Предметы с наименьшим процентом посещаемости (где были пропуски):")
low_attendance = attendance_stats_all[attendance_stats_all['процент_посещений'] < 100].sort_values('процент_посещений')
print(low_attendance.head(10).to_string(index=False))

=== СТАТИСТИКА ПОСЕЩАЕМОСТИ (ВСЕ СТУДЕНТЫ) ===

ТОП-10 предметов по количеству студентов:
 discipline_id                                discipline  уникальных_студентов  всего_записей  количество_посещений  процент_посещений
          3254                          Иностранный язык                  4408          48461                 33770               69.7
         22804       Основы российской государственности                  3413          34009                 17559               51.6
         20329                 Информационные технологии                  3148          30469                 18115               59.5
         17080                                Математика                  3135          50936                 30740               60.4
          4850                 Информационные технологии                  3007          43353                 42847               98.8
          3268                                Математика                  2958          70998       

In [5]:
# 2. СТАТИСТИКА ПО ПОСЕЩАЕМОСТИ ДЛЯ СТУДЕНТОВ С ОЦЕНКАМИ (attendance_filtered)

print("=== СТАТИСТИКА ПОСЕЩАЕМОСТИ (ТОЛЬКО СТУДЕНТЫ С ОЦЕНКАМИ) ===\n")

attendance_stats_filtered = attendance_filtered.groupby(['discipline_id', 'discipline']).agg({
    'mira_id': ['nunique', 'count'],
    'value': lambda x: (x == 1).sum()
}).round(2)

attendance_stats_filtered.columns = ['уникальных_студентов', 'всего_записей', 'количество_посещений']
attendance_stats_filtered = attendance_stats_filtered.reset_index()
attendance_stats_filtered['процент_посещений'] = (
    (attendance_stats_filtered['количество_посещений'] / attendance_stats_filtered['всего_записей'] * 100)
    .round(1)
)

print("Статистика по предметам (студенты с оценками):")
print(attendance_stats_filtered.to_string(index=False))
print("\n")

# Сравнение с общим датасетом
print("СРАВНЕНИЕ ДАТАСЕТОВ:")
print(f"Всего предметов в attendance_all: {attendance_stats_all['discipline_id'].nunique()}")
print(f"Всего предметов в attendance_filtered: {attendance_stats_filtered['discipline_id'].nunique()}")
print(f"Всего студентов в attendance_all: {attendance_all['mira_id'].nunique()}")
print(f"Всего студентов в attendance_filtered: {attendance_filtered['mira_id'].nunique()}")

=== СТАТИСТИКА ПОСЕЩАЕМОСТИ (ТОЛЬКО СТУДЕНТЫ С ОЦЕНКАМИ) ===

Статистика по предметам (студенты с оценками):
 discipline_id                                                                                                                                             discipline  уникальных_студентов  всего_записей  количество_посещений  процент_посещений
          3254                                                                                                                                       Иностранный язык                  4408          48461                 33770               69.7
          3257                                                                                                                                              Философия                   625           5516                  5194               94.2
          3258                                                                                                                                              Эко

In [6]:
# 3. ДОПОЛНИТЕЛЬНАЯ СТАТИСТИКА И АГРЕГАЦИИ

print("=== ДОПОЛНИТЕЛЬНЫЕ АГРЕГАЦИИ ===\n")

# Статистика по студентам (для attendance_all)
student_stats = attendance_all.groupby('mira_id').agg({
    'discipline_id': 'nunique',  # сколько предметов посещал студент
    'lesson_id': 'count',  # сколько всего занятий
    'value': lambda x: (x == 1).sum()  # сколько посещений
}).round(2)

student_stats.columns = ['количество_предметов', 'всего_занятий', 'посещений']
student_stats['процент_посещений_студента'] = (
    (student_stats['посещений'] / student_stats['всего_занятий'] * 100)
    .round(1)
)

print("Статистика по студентам (топ-10 по активности):")
print(student_stats.sort_values('всего_занятий', ascending=False).head(10).to_string())
print("\n")

# Статистика по занятиям (какие уроки самые посещаемые)
lesson_stats = attendance_all.groupby('lesson_id').agg({
    'mira_id': 'nunique',
    'value': lambda x: (x == 1).sum()
}).round(2)

lesson_stats.columns = ['студентов_на_уроке', 'посещений']
lesson_stats['процент_посещений_урока'] = (
    (lesson_stats['посещений'] / lesson_stats['студентов_на_уроке'] * 100)
    .round(1)
)

print("ТОП-10 самых посещаемых уроков:")
print(lesson_stats.sort_values('студентов_на_уроке', ascending=False).head(10).to_string())
print("\n")

# Анализ пропусков (где value пустой или не равен 1)
absences = attendance_all[attendance_all['value'] != 1]
print(f"Всего пропусков: {len(absences)}")
print(f"Студентов с пропусками: {absences['mira_id'].nunique()}")
print(f"Предметов с пропусками: {absences['discipline_id'].nunique()}")

=== ДОПОЛНИТЕЛЬНЫЕ АГРЕГАЦИИ ===

Статистика по студентам (топ-10 по активности):
           количество_предметов  всего_занятий  посещений  процент_посещений_студента
mira_id                                                                              
2440527.0                    13            738        627                        85.0
2442663.0                    13            725        616                        85.0
2439802.0                    13            722        605                        83.8
2450434.0                    13            711        602                        84.7
2445469.0                    13            695        574                        82.6
2463512.0                    21            681        561                        82.4
2440494.0                    13            666        544                        81.7
2459137.0                    17            662        565                        85.3
2452084.0                    17            649        541 

In [7]:
# 4. ОБЪЕДИНЕНИЕ С ТАБЛИЦЕЙ ОЦЕНОК (для будущей модели)

print("=== ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ ===\n")

# Создаем агрегированные данные по студентам из attendance_filtered
student_attendance_agg = attendance_filtered.groupby('mira_id').agg({
    'discipline_id': 'nunique',
    'lesson_id': 'count',
    'value': lambda x: (x == 1).sum()
}).round(2)

student_attendance_agg.columns = ['предметов_посещал', 'всего_занятий', 'всего_посещений']
student_attendance_agg['средняя_посещаемость'] = (
    (student_attendance_agg['всего_посещений'] / student_attendance_agg['всего_занятий'] * 100)
    .round(1)
)

# Агрегируем данные по оценкам для каждого студента
grades_agg = grades.groupby('Student_ID').agg({
    'Discipline_ID': 'nunique',  # сколько предметов
    'Result': lambda x: pd.to_numeric(x, errors='coerce').mean(),  # средний балл (только числовые)
    'Result_ID': lambda x: (x == 6.0).sum()  # количество зачетов (Result_ID=6.0)
}).round(2)

grades_agg.columns = ['предметов_в_оценках', 'средний_балл', 'количество_зачетов']

print("Пример объединенных данных для модели:")
print(grades_agg.head().to_string())
print("\n")



=== ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ ===

Пример объединенных данных для модели:
            предметов_в_оценках  средний_балл  количество_зачетов
Student_ID                                                       
106395                       59          3.75                  16
530187                       99          3.91                  52
608075                       62          3.67                  16
618216                       55          3.83                  14
2234660                      71          3.81                  16




In [8]:
# Создаем сводную таблицу: студент + предмет + количество посещений
student_subject_attendance = attendance_all.groupby(['mira_id', 'discipline_id', 'discipline']).agg({
    'lesson_id': 'count',  # общее количество занятий по предмету
    'value': lambda x: (x == 1).sum()  # количество посещений
}).reset_index()

student_subject_attendance.columns = ['student_id', 'discipline_id', 'discipline_name',
                                      'всего_занятий', 'посещений']

# Добавляем процент посещаемости
student_subject_attendance['процент_посещений'] = (
    (student_subject_attendance['посещений'] / student_subject_attendance['всего_занятий'] * 100)
    .round(1)
)

print(f"Всего записей в разрезе студент-предмет: {len(student_subject_attendance)}")
print("\nПример данных (первые 10 строк):")
print(student_subject_attendance.head(10).to_string())

Всего записей в разрезе студент-предмет: 186113

Пример данных (первые 10 строк):
   student_id  discipline_id                                 discipline_name  всего_занятий  посещений  процент_посещений
0    106395.0           3299                      Теория сварочных процессов              2          0                0.0
1    106395.0           3308                   Сварка специальных материалов              5          5              100.0
2    106395.0           3311      Упрочняющие и восстановительные технологии              6          6              100.0
3    106395.0           3517           Теория решения изобретательских задач              3          3              100.0
4    106395.0          14227     Расчет и проектирование сварных конструкций              4          4              100.0
5    106395.0          17342    Изготовление и сборка изделий машиностроения             10         10              100.0
6    106395.0          20018  Аддитивные технологии в сварочном 